# Эксперимент 2.5: Микропрофилирование вычислительных затрат

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse

from oracles import ExponentialLossL2Oracle
from optimization import nonlinear_conjugate_gradients, hessian_free_newton, lbfgs

In [ ]:
# TODO: подставь реальные данные.
rng = np.random.default_rng(0)
m, n = 3000, 200
A = sparse.csr_matrix(rng.normal(size=(m, n)))
w = rng.normal(size=n)
b = np.sign(A.dot(w) + 0.3 * rng.normal(size=m))
b[b == 0] = 1

def matvec_Ax(x):
    return A.dot(x)
def matvec_ATx(y):
    return A.T.dot(y)
def matmat_ATsA(s):
    return A.T.dot(sparse.diags(s)).dot(A).toarray()

oracle = ExponentialLossL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, b, regcoef=1.0/m)
x0 = np.zeros(n)
ls = {'method': 'Wolfe', 'c1': 1e-4, 'c2': 0.9, 'alpha_0': 1.0}

In [ ]:
# Приближенное профилирование: делим время на части по вызовам.
# Здесь практичная схема через обертки на оракул.
class TimedOracle:
    def __init__(self, base):
        self.base = base
        self.reset()
    def reset(self):
        self.t_func = 0.0
        self.t_grad = 0.0
        self.t_hv = 0.0
    def func(self, x):
        t0 = time.perf_counter(); out = self.base.func(x); self.t_func += time.perf_counter() - t0; return out
    def grad(self, x):
        t0 = time.perf_counter(); out = self.base.grad(x); self.t_grad += time.perf_counter() - t0; return out
    def hess_vec(self, x, v):
        t0 = time.perf_counter(); out = self.base.hess_vec(x, v); self.t_hv += time.perf_counter() - t0; return out
    def func_directional(self, x, d, alpha):
        return self.func(x + alpha * d)
    def grad_directional(self, x, d, alpha):
        return self.grad(x + alpha * d).dot(d)

def profile_method(run_fn, name):
    tor = TimedOracle(oracle)
    t0 = time.perf_counter()
    run_fn(tor)
    total = time.perf_counter() - t0
    oracle_time = tor.t_func + tor.t_grad + tor.t_hv
    algebra_plus_ls = max(total - oracle_time, 0.0)
    # Грубое деление: считаем, что func/grad_directional в line-search уже учтены в oracle_time.
    return {'name': name, 'oracle': oracle_time, 'algo+ls': algebra_plus_ls, 'total': total}

profiles = []
profiles.append(profile_method(lambda o: nonlinear_conjugate_gradients(o, x0, tolerance=1e-6, max_iter=20, line_search_options=ls), 'NLCG'))
profiles.append(profile_method(lambda o: hessian_free_newton(o, x0, tolerance=1e-6, max_iter=10, line_search_options=ls), 'HFN'))
profiles.append(profile_method(lambda o: lbfgs(o, x0, tolerance=1e-6, max_iter=20, memory_size=10, line_search_options=ls), 'L-BFGS'))
profiles

In [ ]:
names = [p['name'] for p in profiles]
oracle_t = [p['oracle'] for p in profiles]
other_t = [p['algo+ls'] for p in profiles]

x = np.arange(len(names))
plt.figure(figsize=(7, 4))
plt.bar(x, oracle_t, label='Oracle (grad/hess_vec/func)')
plt.bar(x, other_t, bottom=oracle_t, label='Linear algebra + line search infra')
plt.xticks(x, names)
plt.ylabel('time, sec')
plt.title('Microprofiling by method')
plt.legend()
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()